# 02b — EDA Stage 1: Cancellation City vs Resort

Cùng trình tự EDA hủy của notebook **02**, nhưng **tách và so sánh** City Hotel vs Resort Hotel.

1. Snapshot volume / tỷ lệ hủy
2. Lead time (bin, KDE, box)
3. Deposit type
4. Market segment
5. Distribution channel + heatmap interaction
6. Gap City − Resort

**Nguồn:** `hotel_bookings_v5.csv` · giữ cả booking hủy.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

NOTEBOOK_DIR = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))
ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "data").is_dir() else NOTEBOOK_DIR
DATA_PATH = ROOT / "data" / "hotel_bookings_v5.csv"
FIG_DIR = ROOT / "reports" / "figures" / "02b"
FIG_DIR.mkdir(parents=True, exist_ok=True)

HOTELS = ["City Hotel", "Resort Hotel"]
HOTEL_COLORS = {"City Hotel": "#4C72B0", "Resort Hotel": "#55A868"}
MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BIN_LABELS = ["0-30", "31-60", "61-90", "91-180", ">180"]
DEPOSIT_ORDER = ["No Deposit", "Non Refund", "Refundable"]
ALPHA = 0.05

print(f"ROOT: {ROOT}")
print(f"DATA: {DATA_PATH}")
print(f"FIG_DIR: {FIG_DIR}")


In [ ]:
def fmt_int(n) -> str:
    return f"{int(round(n)):,}".replace(",", ".")

def fmt_pct(x: float, d: int = 1) -> str:
    return f"{x * 100:.{d}f}%".replace(".", ",")

def fmt_eur(x: float, d: int = 2) -> str:
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"{s} €"

def savefig(name: str) -> Path:
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    print(f"Saved: {path.relative_to(ROOT)}")
    return path

def add_lead_bin(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    edges = [0, 30, 60, 90, 180, float(out["lead_time"].max()) + 1]
    out["lead_time_bin"] = pd.cut(
        out["lead_time"], bins=edges, labels=BIN_LABELS, right=True, include_lowest=True
    )
    return out


In [ ]:
df = pd.read_csv(DATA_PATH, usecols=[
    "hotel", "lead_time", "is_canceled", "deposit_type",
    "market_segment", "distribution_channel",
])
df["hotel"] = pd.Categorical(df["hotel"], categories=HOTELS, ordered=True)
df = add_lead_bin(df)
print(f"Tong booking: {fmt_int(len(df))} | ty le huy: {fmt_pct(df['is_canceled'].mean())}")
for h in HOTELS:
    g = df[df["hotel"] == h]
    print(f"  {h}: n={fmt_int(len(g))} | huy={fmt_pct(g['is_canceled'].mean())} | canceled={fmt_int(g['is_canceled'].sum())}")


## 0. Snapshot so sánh phân khúc


In [ ]:
snap = (
    df.groupby("hotel", observed=True)["is_canceled"]
    .agg(bookings="count", canceled="sum", cancel_rate="mean")
    .reindex(HOTELS)
    .reset_index()
)
display(snap.style.format({"bookings": "{:,.0f}", "canceled": "{:,.0f}", "cancel_rate": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sns.barplot(data=snap, x="hotel", y="bookings", hue="hotel", palette=HOTEL_COLORS, ax=axes[0], legend=False)
axes[0].set_title("So luong booking")
sns.barplot(data=snap, x="hotel", y="cancel_rate", hue="hotel", palette=HOTEL_COLORS, ax=axes[1], legend=False)
axes[1].set_title("Ty le huy")
axes[1].yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
savefig("00_snapshot.png")
plt.show()


## 1. Lead time


In [ ]:
tmp = (
    df.groupby(["hotel", "lead_time_bin", "is_canceled"], observed=True)
    .size().rename("bookings").reset_index()
)
tmp["status"] = tmp["is_canceled"].map({0: "Stay", 1: "Canceled"})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, h in zip(axes, HOTELS):
    sns.barplot(data=tmp[tmp["hotel"]==h], x="lead_time_bin", y="bookings", hue="status",
                hue_order=["Stay","Canceled"], ax=ax)
    ax.set_title(h)
fig.suptitle("Volume theo lead_time bin", y=1.02)
savefig("01_lead_bin_volume.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, h in zip(axes, HOTELS):
    g = df[df["hotel"]==h]
    ct = pd.crosstab(g["lead_time_bin"], g["is_canceled"], normalize="index").reindex(BIN_LABELS)
    ct.plot(kind="bar", stacked=True, ax=ax, color=["#4C72B0","#C44E52"], legend=False)
    ax.set_ylim(0,1); ax.set_title(h)
    ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
axes[1].legend(["Stay","Canceled"])
fig.suptitle("Stacked 100% theo bin", y=1.02)
savefig("02_lead_bin_stacked.png")
plt.show()

rate = df.groupby(["hotel","lead_time_bin"], observed=True)["is_canceled"].mean().reset_index()
display(rate.pivot(index="lead_time_bin", columns="hotel", values="is_canceled").reindex(BIN_LABELS).style.format("{:.1%}"))
fig, ax = plt.subplots(figsize=(10,5))
sns.lineplot(data=rate, x="lead_time_bin", y="is_canceled", hue="hotel",
             hue_order=HOTELS, palette=HOTEL_COLORS, marker="o", ax=ax)
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_title("Ty le huy theo lead_time bin — overlay")
savefig("03_lead_bin_rate_overlay.png")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
for ax, h in zip(axes, HOTELS):
    sns.kdeplot(data=df[df["hotel"]==h], x="lead_time", hue="is_canceled", ax=ax, common_norm=False, clip=(0,400))
    ax.set_title(h); ax.set_xlim(0,400)
fig.suptitle("KDE lead_time", y=1.02)
savefig("04_lead_kde.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
rows = []
for ax, h in zip(axes, HOTELS):
    g = df[df["hotel"]==h]
    sns.boxplot(data=g, x="is_canceled", y="lead_time", showfliers=False, ax=ax)
    ax.set_title(h); ax.set_xticklabels(["Stay","Canceled"])
    for val, lab in [(0,"Stay"),(1,"Canceled")]:
        s = g.loc[g["is_canceled"]==val, "lead_time"]
        rows.append({"hotel":h,"status":lab,"n":len(s),"mean":s.mean(),"median":s.median(),"std":s.std()})
savefig("05_lead_box.png")
plt.show()
display(pd.DataFrame(rows).round(1))


## 2. Deposit type


In [ ]:
dep = (
    df.groupby(["hotel","deposit_type"], observed=True)["is_canceled"]
    .agg(bookings="count", canceled="sum", cancel_rate="mean").reset_index()
)
display(dep.pivot_table(index="deposit_type", columns="hotel", values=["bookings","cancel_rate"]))
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=dep, x="deposit_type", y="cancel_rate", hue="hotel",
            hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_title("Ty le huy theo deposit_type")
savefig("06_deposit_rate.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, h in zip(axes, HOTELS):
    ct = pd.crosstab(df.loc[df["hotel"]==h, "deposit_type"], df.loc[df["hotel"]==h, "is_canceled"], normalize="index").reindex(DEPOSIT_ORDER)
    ct.plot(kind="bar", stacked=True, ax=ax, color=["#4C72B0","#C44E52"], legend=False)
    ax.set_ylim(0,1); ax.set_title(h)
    ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
axes[1].legend(["Stay","Canceled"])
savefig("07_deposit_stacked.png")
plt.show()


## 3. Market segment


In [ ]:
seg = (
    df.groupby(["hotel","market_segment"], observed=True)["is_canceled"]
    .agg(bookings="count", canceled="sum", cancel_rate="mean").reset_index()
)
order_seg = seg.groupby("market_segment")["bookings"].sum().sort_values(ascending=False).index
display(seg.pivot(index="market_segment", columns="hotel", values="cancel_rate").style.format("{:.1%}"))
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=seg, y="market_segment", x="cancel_rate", hue="hotel",
            order=order_seg, hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_title("Ty le huy theo market_segment")
savefig("08_segment_rate.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, h in zip(axes, HOTELS):
    t = seg[seg["hotel"]==h].sort_values("bookings", ascending=False)
    ax2 = ax.twinx()
    ax.bar(t["market_segment"], t["bookings"], color=HOTEL_COLORS[h], alpha=0.85)
    ax2.plot(t["market_segment"], t["cancel_rate"], color="#C44E52", marker="o")
    ax.set_title(h); ax.tick_params(axis="x", rotation=35)
    ax2.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
savefig("09_segment_dual.png")
plt.show()


## 4. Distribution channel + heatmap


In [ ]:
ch = (
    df.groupby(["hotel","distribution_channel"], observed=True)["is_canceled"]
    .agg(bookings="count", canceled="sum", cancel_rate="mean").reset_index()
)
display(ch.pivot(index="distribution_channel", columns="hotel", values="cancel_rate").style.format("{:.1%}"))
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=ch, x="distribution_channel", y="cancel_rate", hue="hotel",
            hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_title("Ty le huy theo distribution_channel")
savefig("10_channel_rate.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, h in zip(axes, HOTELS):
    g = df[df["hotel"]==h]
    mat = pd.crosstab(g["market_segment"], g["distribution_channel"], values=g["is_canceled"], aggfunc="mean")
    sns.heatmap(mat*100, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, vmin=0, vmax=50)
    ax.set_title(h)
fig.suptitle("Cancel rate (%) segment x channel", y=1.02)
savefig("11_heatmap_segment_channel.png")
plt.show()


## 5. Gap City − Resort


In [ ]:
rate = df.groupby(["hotel","lead_time_bin"], observed=True)["is_canceled"].mean().reset_index()
gap = rate.pivot(index="lead_time_bin", columns="hotel", values="is_canceled")
gap["gap_pp"] = (gap["City Hotel"] - gap["Resort Hotel"]) * 100
display(gap)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(gap.index.astype(str), gap["gap_pp"], color="#8172B3")
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Gap ty le huy (City - Resort, diem %)")
ax.set_ylabel("pp")
savefig("12_gap_lead_bin.png")
plt.show()
snap.to_csv(FIG_DIR / "kpi_compare_city_resort.csv", index=False)
